# Module 5.1: Training Loop Walkthrough

## Learning Objectives

By completing this notebook, you will:
1. Implement a complete rollout function
2. Build the GRPO training loop from scratch
3. Track and visualize training progress
4. Understand checkpoint management

## Estimated Time: 15 minutes

---

## 1. The Training Loop

```
for each step:
    for each scenario:
        1. Run N rollouts (agent episodes)
        2. Score with RULER
        3. Compute advantages
        4. Update model (GRPO)
        5. Load new checkpoint
```

We'll implement each piece and simulate the full loop.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass, field

np.random.seed(42)

## 2. Rollout: One Agent Episode

A rollout is a single run of the agent on a task.
The agent receives a question, calls tools, and produces an answer.

In [ ]:
@dataclass
class Trajectory:
    """Record of one agent episode."""
    scenario: str
    messages: list = field(default_factory=list)
    tools_used: list = field(default_factory=list)
    final_answer: str = ""
    num_turns: int = 0
    had_errors: bool = False


def simulate_rollout(scenario: str, skill_level: float) -> Trajectory:
    """
    Simulate an agent rollout.
    
    skill_level: 0.0 (untrained) to 1.0 (expert)
    Higher skill = more likely to explore schema, write correct queries.
    """
    traj = Trajectory(scenario=scenario)
    
    # Decision 1: Explore schema or guess?
    explores_schema = np.random.random() < (0.3 + 0.6 * skill_level)
    
    if explores_schema:
        traj.tools_used.append("list_tables")
        traj.tools_used.append("describe_table")
        traj.num_turns += 2
    
    # Decision 2: Write correct SQL?
    if explores_schema:
        correct_sql = np.random.random() < (0.5 + 0.45 * skill_level)
    else:
        correct_sql = np.random.random() < (0.1 + 0.3 * skill_level)
    
    traj.tools_used.append("run_query")
    traj.num_turns += 1
    
    if not correct_sql:
        traj.had_errors = True
        # Does the agent retry after error?
        retries = np.random.random() < (0.2 + 0.5 * skill_level)
        if retries:
            traj.tools_used.append("run_query")
            traj.num_turns += 1
            correct_sql = np.random.random() < (0.4 + 0.4 * skill_level)
    
    traj.final_answer = "correct" if correct_sql else "incorrect"
    return traj


# Compare untrained vs trained agent
print("Untrained agent (skill=0.1):")
for _ in range(3):
    t = simulate_rollout("Find highest salary", skill_level=0.1)
    print(f"  Tools: {t.tools_used} | Answer: {t.final_answer} | Errors: {t.had_errors}")

print("\nTrained agent (skill=0.9):")
for _ in range(3):
    t = simulate_rollout("Find highest salary", skill_level=0.9)
    print(f"  Tools: {t.tools_used} | Answer: {t.final_answer} | Errors: {t.had_errors}")

## 3. Reward Function

Score trajectories based on correctness, efficiency, and robustness.

In [ ]:
def score_trajectory(traj: Trajectory) -> float:
    """Score a trajectory from 0.0 to 1.0."""
    score = 0.0
    
    # Correctness (50%)
    if traj.final_answer == "correct":
        score += 0.5
    
    # Schema exploration (20%)
    if "describe_table" in traj.tools_used:
        score += 0.2
    
    # Efficiency: fewer turns is better (15%)
    efficiency = max(0, 1 - (traj.num_turns - 2) / 6)  # 2 turns = ideal
    score += 0.15 * efficiency
    
    # Error recovery: if had errors but still got correct answer (15%)
    if traj.had_errors and traj.final_answer == "correct":
        score += 0.15
    elif not traj.had_errors:
        score += 0.1
    
    return min(1.0, score)


# Demo scoring
for skill in [0.1, 0.5, 0.9]:
    scores = [score_trajectory(simulate_rollout("test", skill)) for _ in range(20)]
    print(f"Skill {skill:.1f}: avg={np.mean(scores):.2f}, std={np.std(scores):.2f}")

## 4. Full Training Loop

Run GRPO training and watch the agent improve.

In [ ]:
# Training configuration
GROUP_SIZE = 4
NUM_STEPS = 30
LEARNING_RATE = 0.02

scenarios = [
    "Find the highest-paid employee",
    "Average salary by department",
    "Who leads the most expensive project?",
    "List departments with budget over 1M",
    "Count employees hired in 2023",
]

# Simulated model skill (starts low, improves with training)
skill_level = 0.1

# Training logs
log_rewards = []
log_accuracy = []
log_schema_exploration = []
log_avg_turns = []

for step in range(NUM_STEPS):
    step_rewards = []
    step_correct = 0
    step_explored = 0
    step_turns = []
    total_rollouts = 0
    
    for scenario in scenarios:
        # Run GROUP_SIZE rollouts per scenario
        trajectories = [simulate_rollout(scenario, skill_level) for _ in range(GROUP_SIZE)]
        rewards = [score_trajectory(t) for t in trajectories]
        
        # Compute GRPO advantages
        r = np.array(rewards)
        std = r.std()
        if std > 1e-8:
            advantages = (r - r.mean()) / std
        else:
            advantages = np.zeros_like(r)
        
        # Record metrics
        step_rewards.extend(rewards)
        for t in trajectories:
            step_correct += int(t.final_answer == "correct")
            step_explored += int("describe_table" in t.tools_used)
            step_turns.append(t.num_turns)
            total_rollouts += 1
    
    # Simulate model improvement from GRPO update
    avg_reward = np.mean(step_rewards)
    skill_level = min(0.95, skill_level + LEARNING_RATE * avg_reward)
    
    # Log
    log_rewards.append(avg_reward)
    log_accuracy.append(step_correct / total_rollouts)
    log_schema_exploration.append(step_explored / total_rollouts)
    log_avg_turns.append(np.mean(step_turns))
    
    if (step + 1) % 5 == 0:
        print(
            f"Step {step+1:3d} | "
            f"Reward: {avg_reward:.3f} | "
            f"Accuracy: {log_accuracy[-1]:.0%} | "
            f"Schema: {log_schema_exploration[-1]:.0%} | "
            f"Turns: {log_avg_turns[-1]:.1f}"
        )

In [ ]:
# Visualize training progress
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

axes[0, 0].plot(log_rewards, color="blue")
axes[0, 0].set_title("Average Reward")
axes[0, 0].set_xlabel("Training Step")
axes[0, 0].set_ylabel("Reward")

axes[0, 1].plot(log_accuracy, color="green")
axes[0, 1].set_title("Answer Accuracy")
axes[0, 1].set_xlabel("Training Step")
axes[0, 1].set_ylabel("Accuracy")
axes[0, 1].set_ylim(0, 1)

axes[1, 0].plot(log_schema_exploration, color="orange")
axes[1, 0].set_title("Schema Exploration Rate")
axes[1, 0].set_xlabel("Training Step")
axes[1, 0].set_ylabel("% exploring schema")
axes[1, 0].set_ylim(0, 1)

axes[1, 1].plot(log_avg_turns, color="red")
axes[1, 1].set_title("Average Turns per Rollout")
axes[1, 1].set_xlabel("Training Step")
axes[1, 1].set_ylabel("Turns")

plt.suptitle("GRPO Training Progress", fontsize=14)
plt.tight_layout()
plt.show()

print(f"\nImprovement: Accuracy {log_accuracy[0]:.0%} -> {log_accuracy[-1]:.0%}")
print(f"Schema exploration: {log_schema_exploration[0]:.0%} -> {log_schema_exploration[-1]:.0%}")

## 5. Checkpoint Management

ART saves LoRA checkpoints at intervals. Each checkpoint is a small delta on the base model.

In [ ]:
import json
from pathlib import Path
from datetime import datetime


@dataclass
class Checkpoint:
    step: int
    avg_reward: float
    accuracy: float
    timestamp: str = field(default_factory=lambda: datetime.now().isoformat())


class CheckpointManager:
    """Manage training checkpoints."""
    
    def __init__(self, save_dir: str = "./checkpoints"):
        self.save_dir = Path(save_dir)
        self.checkpoints: list[Checkpoint] = []
    
    def save(self, step: int, avg_reward: float, accuracy: float):
        """Record a checkpoint."""
        ckpt = Checkpoint(step=step, avg_reward=avg_reward, accuracy=accuracy)
        self.checkpoints.append(ckpt)
        return ckpt
    
    def best(self) -> Checkpoint:
        """Return the checkpoint with highest reward."""
        return max(self.checkpoints, key=lambda c: c.avg_reward)
    
    def summary(self):
        """Print checkpoint summary."""
        best = self.best()
        print(f"Total checkpoints: {len(self.checkpoints)}")
        print(f"Best: step {best.step} (reward={best.avg_reward:.3f}, acc={best.accuracy:.0%})")
        print(f"Latest: step {self.checkpoints[-1].step}")


# Simulate checkpoint saving during training
mgr = CheckpointManager()
for i in range(0, NUM_STEPS, 5):
    mgr.save(step=i, avg_reward=log_rewards[i], accuracy=log_accuracy[i])

mgr.summary()

## Exercise: Modify Training Parameters

**Task:** Run the training loop with different parameters and observe the effect.

1. Set GROUP_SIZE to 8 (more diverse trajectories per step)
2. Set LEARNING_RATE to 0.01 (slower, more stable learning)
3. Run for 40 steps
4. Record the final accuracy

In [ ]:
# YOUR CODE HERE
# ---------------

EX_GROUP_SIZE = 0      # TODO: set to 8
EX_LEARNING_RATE = 0   # TODO: set to 0.01
EX_NUM_STEPS = 0       # TODO: set to 40

ex_skill = 0.1
ex_final_accuracy = 0.0

# TODO: Implement the training loop with these parameters
# Use the same simulate_rollout and score_trajectory functions
# Store the final accuracy in ex_final_accuracy

exercise_group_size = EX_GROUP_SIZE
exercise_lr = EX_LEARNING_RATE
exercise_steps = EX_NUM_STEPS
exercise_accuracy = ex_final_accuracy

In [ ]:
# AUTO-GRADED TESTS - Do not modify
# ----------------------------------

def test_exercise():
    assert exercise_group_size == 8, f"Expected group_size=8, got {exercise_group_size}"
    assert exercise_lr == 0.01, f"Expected lr=0.01, got {exercise_lr}"
    assert exercise_steps == 40, f"Expected 40 steps, got {exercise_steps}"
    assert exercise_accuracy > 0.3, f"Final accuracy should be > 30%, got {exercise_accuracy:.0%}"
    print(f"All tests passed! Final accuracy: {exercise_accuracy:.0%}")

test_exercise()

## Summary

**Key Takeaways:**
1. A rollout is a single agent episode: question → tool calls → answer
2. The training loop: rollouts → scoring → advantages → GRPO update → new checkpoint
3. Training improves accuracy, schema exploration rate, and query efficiency
4. Checkpoints let you evaluate the model at different stages

**Next:** Module 06 puts it all together with a real text-to-SQL agent.